#### Requirements
https://share.gemini.google/OyWKQhkPAz0g

In [373]:
import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st
import plotly.graph_objects as go

In [374]:
df = pd.read_csv('overall_funding_in_india.csv')

In [375]:
df['date']= pd.to_datetime(df['date'],dayfirst=False)
df['amount']= df['amount'].astype(float)

In [376]:
df = df[~(df['amount'] == 0)]

In [377]:
df["city"] = (
    df["city"]
    .str.replace("\xa0", " ", regex=False)       # non-breaking spaces
    .str.replace(r"\s+", " ", regex=True)        # collapse repeated spaces
    .str.strip()                                 # leading/trailing spaces
)

## unique cities invested

In [378]:
cities_invested = sorted(df['city'][~(df['city'].str.contains('india',case=False))].unique())

In [379]:
city_rep = df['city'][(~df['city'].str.contains('india', case=False))].value_counts().reset_index().head(10)
fig = px.pie(city_rep , names='city' , values='count' ,hole=0.4,color_discrete_sequence=px.colors.sequential.Aggrnyl)
fig.update_layout(annotations=[
    {'text':'<b>Total Funded Cities</b><br>{}'.format(len(cities_invested)),'x':0.5 ,'y':0.5 ,'showarrow':False ,'font_size':11}
] , paper_bgcolor='black' ,font_color='white')
fig.show()

In [380]:
# TOTAL AMOUNT INVESTED
df['amount'].sum()

np.float64(64740684986.0)

In [381]:
### highest single funding
df['amount'].max()

np.float64(3900000000.0)

#### highest funding in a startup

In [382]:
df.groupby('startup')['amount'].sum().sort_values(ascending=False).head(1).values[0]

np.float64(5001448000.0)

In [383]:
# TOTAL UNIQUE STARTUPS
df['startup'].nunique()

1541

In [384]:
np.mean(df['amount'])   # average

np.float64(21366562.701650165)

In [385]:
df['year'] = df['date'].dt.year

### YOY NO. OF STARTUPS

In [386]:
no_of_startups= df.groupby('year')['startup'].count().reset_index()
trace1 = go.Scatter(x=no_of_startups['year'] ,y=no_of_startups['startup'])
data = [trace1]
layout = go.Layout(title = {'text':'<b>Number of Startups Funded Per Year</b>','x':0.5
                         }
                         ,xaxis={'title':'<b>Year</b>' ,'dtick':1} ,yaxis={'title':'<b>Number of Startups</b>'})
fig = go.Figure(data ,layout)
fig.show()

### YOY amount if investment

In [387]:
amount_invs_per_year= df.groupby('year')['amount'].sum().reset_index()

trace1 = go.Scatter(x=amount_invs_per_year['year'] ,y=amount_invs_per_year['amount'] ,marker={'color':'red'})
data = [trace1]
layout = go.Layout(title = {'text':'<b>Total Investment per Year</b>','x':0.5
                         }
                         ,xaxis={'title':'<b>Year</b>' ,'dtick':1} ,yaxis={'title':'<b>Amount</b>'})
fig = go.Figure(data ,layout)
fig.show()

In [388]:
## to compare startups
def startup_plot_line(*args):
    data = list()
    for i in args:
        temp_ser = df[df['startup'] == i]
        temp_ser = temp_ser.groupby('year')['amount'].sum()
        trace = go.Scatter(x=temp_ser.index ,y =temp_ser.values ,name=i ,
                            mode='lines+markers',              
                            line=dict(width=3, shape='spline'), 
                            marker=dict(size=8, symbol='circle') )
        data.append(trace)
    return data 

In [389]:
data = startup_plot_line('Ola','Udaan','Flipkart')
layout = go.Layout(title = {'text':'<b>Year Over Year Startups Comparison</b>',
                            'x':0.5 },
                   
                   xaxis=dict(
                           title='<b>Year</b>',
                           dtick=1,                                # Forces the x-axis to show only whole years
                           showgrid=False,                         # Removes vertical gridlines for a cleaner look
                           linecolor='black'
                       ),yaxis={'title':'<B>Amount</b>'},
                       hovermode='x unified'                   # Shows a single hover box for all startups in a given year
                       )
fig = go.Figure(data ,layout)
fig.show()

### top 10 verticals

In [390]:
top_10_vericals = df.groupby('vertical')['amount'].sum().sort_values(ascending=False).reset_index().head(10)

In [391]:
trace1 = go.Bar(x=top_10_vericals['vertical'] ,y=top_10_vericals['amount'],marker=dict(
                                                                        color=top_10_vericals['amount'], 
                                                                        colorscale='RdBu'))
data = [trace1]
layout = go.Layout(title = {'text':'<b>Top 10 Sectors</b>','x':0.5}
                         ,xaxis={'title':'<b>Sectors</b>'} ,yaxis={'title':'<b>Amount</b>'})
fig = go.Figure(data ,layout)
fig.show()

In [392]:
df['month'] = df['date'].dt.month

In [393]:
# heat map
df.pivot_table(index='month' ,columns='year' ,values='amount',aggfunc='sum',fill_value=0)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
month,,,,,,,,,,,
1,0.000000e+00,699880000.0,2.532335e+08,3.473746e+08,1.370000e+08,1.044680e+09,8.840400e+07,206732000.0,188820000.0,7.356080e+08,687900000.0
2,0.000000e+00,331490500.0,2.949590e+08,1.211217e+09,2.266000e+08,5.481050e+08,8.088000e+07,584668000.0,157788000.0,1.292731e+09,748345000.0
3,0.000000e+00,309561608.0,2.264281e+09,2.919530e+08,0.000000e+00,6.456180e+08,9.006330e+08,91483000.0,383470000.0,3.369330e+08,194028000.0
4,9.805000e+08,253940000.0,7.323950e+08,1.980777e+08,1.301354e+08,7.781420e+08,5.185240e+08,261871000.0,236863000.0,8.248460e+08,346552000.0
5,3.201350e+08,339840000.0,1.786669e+09,1.058870e+08,4.089686e+08,2.451000e+08,5.939030e+08,796631000.0,304530000.0,3.354190e+08,705665000.0
6,5.219940e+08,199639000.0,3.448180e+08,3.914750e+08,2.895237e+08,4.887210e+08,1.058510e+08,744198000.0,11429000.0,1.746160e+08,130725000.0
7,1.580427e+09,233547000.0,1.677040e+08,5.519780e+08,2.967946e+08,1.140820e+08,3.781320e+08,77563000.0,232560000.0,4.896980e+08,0.0
8,1.063816e+09,534089000.0,2.748139e+09,8.093548e+08,4.531770e+09,4.657500e+08,5.235970e+08,253488000.0,396741000.0,5.554040e+08,0.0
9,1.470520e+09,241361000.0,5.365020e+08,6.010900e+08,5.028000e+08,3.752900e+08,1.000908e+09,110208000.0,909027000.0,1.474430e+08,0.0


## unique cities

In [394]:
temp_df= df[~(df['city'].str.contains('india',case=False))]

In [395]:
temp_df = temp_df.groupby('city',as_index=False)['amount'].sum()

In [396]:
lst = df.groupby('city')['amount'].sum().sort_values(ascending=False).head(10).index.tolist()
temp_df = df[df['city'].isin(lst)]


### top 10 cities with top 10 sectors of each city with subvertical

In [397]:
temp_df = temp_df.sort_values(by=['city','amount'],ascending=[True,False]).groupby('city').head(10)

In [398]:
fig = px.sunburst(temp_df,path=['city','vertical','sub vertical'] ,values='amount')
fig.update_traces(
    hovertemplate=(
        "<b>%{label}</b><br>" +
        "Funding: $%{value:,.0f}<br>" +          # Adds commas (e.g., $3,900,000,000)
        "Share of Parent: %{percentParent:.1%}<br>" +
        "Share of Total: %{percentRoot:.1%}" +
        "<extra></extra>"                         # Removes the secondary trace box
    )
)
fig.show()

### total unique investors

In [399]:
investors = pd.Series(df['investors'].str.strip().str.split(',').sum())
investors = investors.str.strip()

#### no of unique investors

In [400]:
no_of_unique_investors = investors.nunique()

In [401]:
# investors with number of investments
Investor_with_noOfInves = investors.value_counts()

In [402]:
Investor_with_noOfInves [Investor_with_noOfInves>10]

Undisclosed Investors        148
Kalaari Capital              127
Blume Ventures               110
Info Edge                    101
Nexus Venture Partners       100
Tiger Global Management       95
Y Combinator                  94
Sequoia Capital India         92
Tiger Global                  90
Omidyar Network               90
Accel                         85
Mirae Asset                   83
IFC                           81
Bessemer Venture Partners     79
Zodius Capital                77
Ribbit Capital                76
Prosus Ventures               75
SoftBank Vision Fund          72
Kedaara Capital               70
Accel Partners                69
Peak XV                       69
Matrix Partners India         68
Falcon Edge                   68
Elevation Capital             67
A91 Partners                  65
Ventures India                65
LCatterton                    64
Sequoia Capital               62
Lightspeed India              58
SAIF Partners                 51
Indian Ang

### names for user in sorted form 

In [403]:
sorted(investors.unique())

['021 Capita',
 '100 Co-founders Lab',
 '122 West Ventures',
 '1Crowd',
 '3ONE4 Capital Advisors',
 '3one4 Capital',
 '500 Startups',
 '50K Ventures',
 '91SpringBoard',
 '91maker',
 'A M Sikander',
 'A91 Partners',
 'AB Kinnevik',
 'ABB Technology Ventures',
 'ABCOM Investments',
 'ABI-Showatech (India)',
 'ABM Knowledgeware',
 'ADAG Reliance Private Equity',
 'AECAL',
 'AET Fund',
 'AJ Ventures',
 'AL Nour International Holdings',
 'ANI Technologies',
 'APIS Partners',
 'ASK Pravi',
 'ASP Consulting',
 'Aamir Khan',
 'Aarin Capital',
 'Aarin Capital Partners',
 'Aarti Group',
 'Aavishkaar',
 'Aavishkaar Venture Management',
 'Abhai S. Rao',
 'Abhay Singhal',
 'Abhinav Bindra',
 'Abhinav Gupta',
 'Abhinav Patodia',
 'Abhishek Acharya',
 'Abhishek Agarwal',
 'Abhishek Goyal',
 'Abhishek Gupta',
 'Abhishek Jain',
 'Abraaj Group',
 'Abstract Ventures',
 'Abu Farhan Azmi',
 'Accel',
 'Accel India',
 'Accel Partners',
 'Accel Partners India',
 'Accel Partners RNT Associates',
 'Accel US',
 

In [404]:
temp_ind  = pd.read_csv('temp_india.csv')

### total Undisclosed Deals

In [405]:
np.round((924/df.shape[0])*100 ,2)

np.float64(30.5)

### heatmap

In [430]:
heat_df = df.pivot_table(index='year',columns='month' ,values='amount')

In [436]:
fig = go.Figure(
    data=go.Heatmap(
        z=heat_df.values,
        x=heat_df.columns,
        y=heat_df.index,
        colorscale="Blues",
        # 1. Round numbers to 1 decimal place (e.g., 5.7M instead of 5.663944M)
        texttemplate="%{z:.2s}",
        textfont=dict(size=11),
        # 2. Cap the max color scale so outliers (315M) don't wash out the rest
        zmin=0,
        zmax=100_000_000,  # Adjust this threshold based on your data (e.g., 90th percentile)
        # 3. Add crisp white borders between every cell
        xgap=2,
        ygap=2,
        hoverongaps=False,
        hovertemplate="Month: %{x}<br>Year: %{y}<br>Value: %{z:,.0f}<extra></extra>",
    )
)

fig.update_layout(
    title="Monthly Performance Heatmap",
    # 4. Force every Month (1-12) and Year to display cleanly as categories
    xaxis=dict(
        title="Month",
        type="category",
        side="top",  # Optional: puts month numbers at the top like a table
    ),
    yaxis=dict(
        title="Year",
        type="category",
        autorange="reversed",  # Keeps 2026 at the top
    ),
    plot_bgcolor="white",  # Makes NaN cells clean white instead of grey-blue
)

fig.show()